# Lily 1.5B GRPO Model Inference — Google Colab
**Interactive Reasoning & Inference for `abhinav0231/Lily-1.5B`**

This notebook runs in Google Colab. It loads the trained GRPO model `abhinav0231/Lily-1.5B` using Unsloth's 2x fast inference engine (`FastLanguageModel.for_inference`), parses step-by-step thinking `<think>...</think>` tags and final `<answer>...</answer>` tags, and includes an interactive query loop for testing.

## Cell 1 — Install Unsloth & Dependencies

In [ ]:
!pip install unsloth huggingface_hub -q

## Cell 2 — Load Trained GRPO Model with Unsloth

In [ ]:
import os
from unsloth import FastLanguageModel
from huggingface_hub import login

# Hugging Face Authentication
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    login(token=HF_TOKEN)

HF_USERNAME = "abhinav0231"
MODEL_REPO  = f"{HF_USERNAME}/Lily-1.5B"  # Trained GRPO model
MAX_SEQ_LEN = 3072

print(f"Loading trained GRPO model: {MODEL_REPO} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,             # Auto-detects float16 / bfloat16
    load_in_4bit   = True,
    token          = HF_TOKEN if HF_TOKEN else None,
)

# Enable Unsloth 2x fast inference mode
FastLanguageModel.for_inference(model)
print("\n✅ Model loaded successfully and configured for fast inference!")

## Cell 3 — Inference Function with Reasoning CoT Parser

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

def ask(question, max_new_tokens=1024, temperature=0.7):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    output_ids = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = 0.95,
        do_sample      = True if temperature > 0 else False,
        pad_token_id   = tokenizer.eos_token_id,
    )

    # Decode only the newly generated tokens
    response = tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:],
        skip_special_tokens = True
    )
    return response

def parse_and_print(response):
    think_m  = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)
    
    print("🧠 REASONING (<think>):")
    if think_m:
        print(think_m.group(1).strip())
    else:
        print("No <think> tag found. Full output:")
        print(response)
        
    print("\n🎯 FINAL ANSWER (<answer>):")
    if answer_m:
        print(answer_m.group(1).strip())
    else:
        print(response.split("</think>")[-1].strip())

print("✅ Inference & Parsing functions ready")

## Cell 4 — Run Sample Test Queries (Math, Logic, Coding)

In [ ]:
test_questions = [
    "What is 15% of 840?",
    "If a train travels 120 km in 1.5 hours, what is its speed in m/s?",
    "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much does the ball cost?",
    "Write a Python function to check if a string is a palindrome."
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"❓ QUESTION: {q}")
    print(f"{'='*70}")
    raw_out = ask(q, temperature=0.7)
    parse_and_print(raw_out)

## Cell 5 — Interactive Query Testing Loop

In [ ]:
print("Type your question below (or type 'exit' to stop):\n")
while True:
    user_query = input("Enter Query: ")
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("Exiting interactive loop.")
        break
    if not user_query.strip():
        continue
        
    print(f"\n{'='*70}")
    print(f"❓ QUESTION: {user_query}")
    print(f"{'='*70}")
    raw_out = ask(user_query, temperature=0.7)
    parse_and_print(raw_out)
    print("\n")